# **Day 95: Building Web Apps with Flask**

**Flask** is a small and easy **web framework** for Python. With it, we can make websites and APIs.

Install it with `pip install flask`.

**How a website works:**
1. The **browser** sends a **request** to a URL (like `/districts`).
2. The **server** (our Python code) runs a function for that URL.
3. The function returns a **response** (a web page or JSON data).

A running server keeps the notebook busy, so we write the app into a file (`app.py`), and test it here using Flask's **test client**, which sends pretend requests. To run it for real, type `python app.py` in the terminal and open `http://127.0.0.1:5000` in the browser.

In [1]:
%%writefile app.py
from flask import Flask, request, render_template_string, abort, jsonify

app = Flask(__name__)

DISTRICTS = {
    "kolkata": {"name": "Kolkata", "hq": "Kolkata", "lat": 22.5726, "lon": 88.3639, "population": 4496694},
    "darjeeling": {"name": "Darjeeling", "hq": "Darjeeling", "lat": 27.0410, "lon": 88.2663, "population": 1846823},
    "paschim-medinipur": {"name": "Paschim Medinipur", "hq": "Midnapore", "lat": 22.4257, "lon": 87.3199, "population": 5913457},
    "purulia": {"name": "Purulia", "hq": "Purulia", "lat": 23.3321, "lon": 86.3652, "population": 2930115},
}

LIST_PAGE = """
<h1>Districts of West Bengal</h1>
<form action="/search"><input name="q" placeholder="Search"><button>Go</button></form>
<ul>
{% for slug, d in districts.items() %}
  <li><a href="/district/{{ slug }}">{{ d.name }}</a> (HQ: {{ d.hq }})</li>
{% endfor %}
</ul>
"""

@app.route("/")
def home():
    return "<h1>Nomoskar!</h1><p>Go to <a href='/districts'>/districts</a></p>"

@app.route("/districts")
def district_list():
    return render_template_string(LIST_PAGE, districts=DISTRICTS)

@app.route("/district/<slug>")
def district_page(slug):
    d = DISTRICTS.get(slug)
    if d is None:
        abort(404)
    return (f"<h2>{d['name']}</h2><p>Headquarters: {d['hq']}</p>"
            f"<p>Location: {d['lat']}, {d['lon']}</p><p>Population (2011): {d['population']:,}</p>")

@app.route("/search")
def search():
    q = request.args.get("q", "").strip().lower()
    found = {k: v for k, v in DISTRICTS.items() if q in v["name"].lower() or q in v["hq"].lower()}
    return render_template_string(LIST_PAGE, districts=found)

@app.route("/api/distance")
def distance_api():
    import math
    a = DISTRICTS.get(request.args.get("from", ""))
    b = DISTRICTS.get(request.args.get("to", ""))
    if not a or not b:
        return jsonify(error="unknown district"), 400
    p1, p2 = math.radians(a["lat"]), math.radians(b["lat"])
    dl = math.radians(b["lon"] - a["lon"])
    h = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    km = 2 * 6371 * math.asin(math.sqrt(h))
    return jsonify(frm=a["name"], to=b["name"], straight_line_km=round(km, 1))

@app.errorhandler(404)
def not_found(e):
    return "<h2>404 - This page does not exist!</h2>", 404

if __name__ == "__main__":
    app.run(debug=True)

Writing app.py


## **Testing the App**

In [2]:
from app import app

client = app.test_client()

r = client.get("/")
print(r.status_code, r.content_type)
print(r.data.decode())

200 text/html; charset=utf-8
<h1>Nomoskar!</h1><p>Go to <a href='/districts'>/districts</a></p>


In [3]:
print(client.get("/districts").data.decode())


<h1>Districts of West Bengal</h1>
<form action="/search"><input name="q" placeholder="Search"><button>Go</button></form>
<ul>

  <li><a href="/district/kolkata">Kolkata</a> (HQ: Kolkata)</li>

  <li><a href="/district/darjeeling">Darjeeling</a> (HQ: Darjeeling)</li>

  <li><a href="/district/paschim-medinipur">Paschim Medinipur</a> (HQ: Midnapore)</li>

  <li><a href="/district/purulia">Purulia</a> (HQ: Purulia)</li>

</ul>


In [4]:
print(client.get("/district/paschim-medinipur").data.decode())

<h2>Paschim Medinipur</h2><p>Headquarters: Midnapore</p><p>Location: 22.4257, 87.3199</p><p>Population (2011): 5,913,457</p>


In [5]:
r = client.get("/district/goa")
print(r.status_code, r.data.decode())

404 <h2>404 - This page does not exist!</h2>


In [6]:
r = client.get("/search", query_string={"q": "midnapore"})
print(r.data.decode())


<h1>Districts of West Bengal</h1>
<form action="/search"><input name="q" placeholder="Search"><button>Go</button></form>
<ul>

  <li><a href="/district/paschim-medinipur">Paschim Medinipur</a> (HQ: Midnapore)</li>

</ul>


## **A Small JSON API**
The `/api/distance` route returns **JSON** instead of HTML. Other programs (like a mobile app or a map website) can use it.

In [7]:
r = client.get("/api/distance", query_string={"from": "kolkata", "to": "darjeeling"})
print(r.status_code, r.get_json())

r = client.get("/api/distance", query_string={"from": "kolkata", "to": "mars"})
print(r.status_code, r.get_json())

200 {'frm': 'Kolkata', 'straight_line_km': 497.0, 'to': 'Darjeeling'}
400 {'error': 'unknown district'}


## **Important Ideas in Flask**
| Code | Meaning |
|---|---|
| `@app.route("/path")` | Connects a URL to a function (a decorator, Day 57!) |
| `<slug>` in the path | A part of the URL that becomes a function argument |
| `request.args` | Values after `?` in the URL (`?q=kolkata`) |
| `render_template_string()` | Makes HTML using the **Jinja2** template language |
| `{{ value }}` | Shows a value inside the HTML |
| `{% for ... %}` | A loop inside the HTML |
| `jsonify()` | Returns JSON data |
| `return body, 404` | Returns a status code too |

In real projects, HTML templates are kept in a separate `templates/` folder, and we use `render_template("page.html")`.

In [8]:
# Quick question
# Which URL would show the page for Darjeeling in this app?